In [1]:
# Run this inside your main.ipynb

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report # Imported accuracy_score
from imblearn.over_sampling import SMOTE
import joblib

# 1. Load Data
df = pd.read_csv('dataset.csv')

# 2. Feature Selection & Encoding
# We only keep features present in the index.html form
# Dropping: Patient_ID, Created_On, Age, Community_Code, Region_Code

# Map binary categorical variables
binary_map = {'Yes': 1, 'No': 0}
df['Is_Parent_Carrier'] = df['Is_Parent_Carrier'].map(binary_map)
df['Is_Sibling_Affected'] = df['Is_Sibling_Affected'].map(binary_map)
df['Has_Chronic_Cough'] = df['Has_Chronic_Cough'].map(binary_map)

# Map Newborn Screen
df['Newborn_Screen_Flag'] = df['Newborn_Screen_Flag'].map({'Detected': 1, 'Not_Detected': 0})

# Map Gender directly (1 for Male, 0 for Female to match HTML values)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# Encode Target Variable
le = LabelEncoder()
df['Disease_Status'] = le.fit_transform(df['Disease_Status'])

# Define the exact list of features to use (Must match index.html inputs)
features = [
    'Maternal_Age_at_Birth', 
    'Gender', 
    'Hemoglobin_Level_g/dL', 
    'MCV_fL', 
    'MCH_pg', 
    'Family_History_Score', 
    'Is_Parent_Carrier', 
    'Is_Sibling_Affected', 
    'Has_Chronic_Cough', 
    'Newborn_Screen_Flag'
]

X = df[features]
y = df['Disease_Status']

# 3. Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize and fit model
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_resampled, y_train_resampled)

# 4. Evaluate (Get Accuracy)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# 5. Save
joblib.dump(model, 'genetic_model.joblib')
joblib.dump(le, 'label_encoder.joblib')
print("Model and Encoder saved successfully!")

Model Accuracy: 85.04%

Classification Report:
                      precision    recall  f1-score   support

   Beta_Thalassemia       0.69      0.80      0.74       501
    Cystic_Fibrosis       0.03      0.08      0.05        96
      Down_Syndrome       0.04      0.09      0.06       207
            Healthy       0.96      0.90      0.93      8804
Huntingtons_Disease       0.00      0.00      0.00        97
 Sickle_Cell_Anemia       0.56      0.42      0.48       295

           accuracy                           0.85     10000
          macro avg       0.38      0.38      0.38     10000
       weighted avg       0.89      0.85      0.87     10000

Model and Encoder saved successfully!
